# LightGBM with three seeds
Run all cells from the project directory. Compare the seed-42 model with its three-seed average. All predictions use a threshold of 0.5.

In [1]:
%pip install lightgbm pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'raw').is_dir())  # project root
DATA = ROOT / 'data' / 'raw'
SUBMISSIONS = ROOT / 'submissions'
train = pd.read_csv(DATA / 'train.csv')
test = pd.read_csv(DATA / 'test.csv')
sample = pd.read_csv(DATA / 'sample_submission.csv')
y = train.pop('Transported').astype(int)
rows = pd.concat([train, test], ignore_index=True)
assert sample.PassengerId.equals(test.PassengerId) and rows.PassengerId.is_unique

## Passenger and cabin features
The features use cabin occupancy, age and cabin bands, spending bands, and passenger groups. Group counts and unanimous group attributes use the complete unlabeled passenger manifest. Deck and side are used directly, not as deck transport rates, because rates calculated before splitting for validation would expose validation labels.

In [3]:
SPEND = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]


def features(rows):
    x = rows.copy()
    group = x.pop("PassengerId").str.split("_").str[0]
    family = x.pop("Name").str.split().str[-1]
    x[["Deck", "Number", "Side"]] = x.pop("Cabin").str.split("/", expand=True)
    x["Number"] = pd.to_numeric(x.Number)
    x["GroupNumber"] = group.astype(int)
    x["GroupSize"] = group.groupby(group).transform("size")
    x["FamilySize"] = family.groupby(family).transform("size")
    for col in ["Deck", "Side", "Number", "HomePlanet"]:
        matches = x[col].groupby(group)
        unanimous = matches.transform("first").where(matches.transform("nunique") == 1)
        x[col] = x[col].fillna(unanimous)
    x["TotalSpend"] = x[SPEND].sum(axis=1, min_count=1)
    x["MissingSpend"] = x[SPEND].isna().sum(axis=1)
    x["GroupMeanSpend"] = x.TotalSpend.groupby(group).transform("mean")
    x.loc[x.CryoSleep.eq(True), SPEND] = x.loc[x.CryoSleep.eq(True), SPEND].fillna(0)
    for col in ["Deck", "Side", "HomePlanet", "Destination", "CryoSleep", "VIP"]:
        x[col] = x[col].astype("string").fillna("Unknown").astype("category")
    return x

In [4]:
x = features(rows)
x['CabinMembers'] = rows.groupby('Cabin').PassengerId.transform('size')
for source, cuts in {'Age': [17, 43], 'Number': [300, 690, 1170],
                     'GroupNumber': [3400, 7300], 'TotalSpend': [1, 2250]}.items():
    x[source + 'Band'] = pd.cut(x[source], [-np.inf, *cuts, np.inf], labels=False)



In [5]:
folds = list(StratifiedKFold(5, shuffle=True, random_state=42).split(x.iloc[:len(y)], y))

In [6]:
inputs = pd.get_dummies(x, columns=x.select_dtypes('category').columns, dtype=float)
seeds = (42, 73, 101)

def make_model(name, seed):
    params = dict(n_estimators=550, learning_rate=.03, random_state=seed, n_jobs=4,
                  subsample=.85, colsample_bytree=.9)
    return LGBMClassifier(**params, num_leaves=15, subsample_freq=1, verbosity=-1)

def evaluate(name, seed):
    oof = np.full(len(y), np.nan)
    predictions = []
    for tr, va in folds:
        model = make_model(name, seed)
        model.fit(inputs.iloc[tr], y.iloc[tr])
        oof[va] = model.predict_proba(inputs.iloc[va])[:, 1]
        predictions.append(model.predict_proba(inputs.iloc[len(y):])[:, 1])
    assert np.isfinite(oof).all() and np.isfinite(predictions).all()
    print(name, seed, 'finished', flush=True)
    return oof, np.mean(predictions, axis=0)

In [7]:
validation = pd.DataFrame(index=y.index)
predictions = pd.DataFrame(index=test.index)
for name in ['LightGBM']:
    results = [evaluate(name, seed) for seed in seeds]
    validation[name + ' seed 42'] = results[0][0]
    validation[name] = np.mean([v for v, _ in results], axis=0)
    predictions[name] = np.mean([p for _, p in results], axis=0)

correct = validation.ge(.5).eq(y, axis=0)
scores = pd.DataFrame([correct.iloc[va].mean() for _, va in folds], index=range(1, 6))
scores.loc['Overall'] = correct.mean()
display(scores.round(5))

LightGBM 42 finished


LightGBM 73 finished


LightGBM 101 finished


,LightGBM seed 42,LightGBM
1,0.81254,0.81254
2,0.81254,0.81656
3,0.81599,0.81771
4,0.80955,0.80840
5,0.80495,0.80322
Overall,0.81111,0.81169


In [8]:
for name, suffix in [('LightGBM', 'lightgbm')]:
    output = sample.copy()
    output['Transported'] = predictions[name].ge(.5).to_numpy()
    path = SUBMISSIONS / f'submission_{suffix}_seeds.csv'
    output.to_csv(path, index=False)
    print(path, len(output), 'passengers')

C:\Users\shaya\titanic-spaceShip\submissions\submission_lightgbm_seeds.csv 4277 passengers
